<a href="https://colab.research.google.com/github/nikita-porwal-git/Property-Price-Prediction-Linear-Regression-/blob/main/property_price_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ==========================================
# Nikita Porwal | Data Science Portfolio
# Project: Property Price Prediction using Linear Regression
# ==========================================

# ===============================
# IMPORT LIBRARIES
# ===============================
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error

import matplotlib.pyplot as plt
import seaborn as sns

# ===============================
# LOAD DATA
# ===============================
# Upload files in Colab:
# Property_Price_Train.csv
# Property_Price_Test.csv

from google.colab import files
uploaded = files.upload()

train = pd.read_csv("Property_Price_Train.csv")
test = pd.read_csv("Property_Price_Test.csv")

print(" Data Loaded Successfully")

# ===============================
# DATA UNDERSTANDING
# ===============================
print("Train Shape:", train.shape)
print("Test Shape:", test.shape)

# ===============================
# MERGE DATA
# ===============================
train['Sample'] = 'train'
test['Sample'] = 'test'

data = pd.concat([train, test], axis=0, sort=False)

# ===============================
# DATA CLEANING
# ===============================
# Fill missing values (simplified version for clean notebook)
data.fillna(method='ffill', inplace=True)

print(" Missing values handled")

# ===============================
# FEATURE ENGINEERING
# ===============================
# Example: Total Area Feature
data['Total_Area'] = (
    data.get('Total_Basement_Area', 0) +
    data.get('First_Floor_Area', 0) +
    data.get('Second_Floor_Area', 0)
)

# ===============================
# ENCODING
# ===============================
categorical_cols = data.select_dtypes(include=['object']).columns

for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))

print("✅ Encoding Done")

# ===============================
# SPLIT BACK TRAIN & TEST
# ===============================
train_data = data[data['Sample'] == 1].copy()
test_data = data[data['Sample'] == 0].copy()

# Remove Sample column
train_data.drop(columns=['Sample'], inplace=True)
test_data.drop(columns=['Sample'], inplace=True)

# Target variable
y = train_data['Sale_Price']
X = train_data.drop(columns=['Sale_Price'])

# Scale data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y, test_size=0.1, random_state=42
)

print("Data Split Done")

# ===============================
# LINEAR REGRESSION
# ===============================
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

print("\n Linear Regression MAE:")
print(mean_absolute_error(y_val, y_pred))

# ===============================
# RIDGE REGRESSION
# ===============================
ridge_params = {"alpha": np.linspace(1, 100, 10)}

ridge = GridSearchCV(Ridge(), ridge_params, cv=5, scoring="neg_mean_absolute_error")
ridge.fit(X_train, y_train)

y_pred_ridge = ridge.predict(X_val)

print("\n Ridge Regression MAE:")
print(mean_absolute_error(y_val, y_pred_ridge))

# ===============================
# LASSO REGRESSION
# ===============================
lasso_params = {"alpha": np.linspace(0.001, 0.1, 10)}

lasso = GridSearchCV(Lasso(), lasso_params, cv=5, scoring="neg_mean_absolute_error")
lasso.fit(X_train, y_train)

y_pred_lasso = lasso.predict(X_val)

print("\n Lasso Regression MAE:")
print(mean_absolute_error(y_val, y_pred_lasso))

# ===============================
# VISUALISATION
# ===============================
plt.figure(figsize=(6,4))
sns.scatterplot(x=y_val, y=y_pred)
plt.xlabel("Actual Prices")
plt.ylabel("Predicted Prices")
plt.title("Actual vs Predicted Prices")
plt.show()

# ===============================
# CONCLUSION
# ===============================
print("""
Conclusion:
✔ Built regression models for property price prediction
✔ Compared Linear, Ridge, and Lasso models
✔ Ridge/Lasso help reduce overfitting
✔ Feature engineering improves model performance
""")